<a href="https://colab.research.google.com/github/g25ait1051-collab/g25ait1051-iitj.ac.in/blob/main/NLU_Assignmnet_1_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 2: Language Model

In [ ]:
import nltk
from collections import defaultdict, Counter
import random
import math

# Download necessary NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')
# Download Gutenberg corpus for this part
try:
    nltk.data.find('corpora/gutenberg')
except LookupError:
    nltk.download('gutenberg')

# --- 0. Corpus Loading and Preprocessing ---

# Load a text from the Project Gutenberg corpus
# You can choose a different file if needed, e.g., 'carroll-alice.txt'
raw_corpus = nltk.corpus.gutenberg.raw('austen-emma.txt')

print("--- Corpus Preprocessing ---")
print(f"Raw Corpus Length: {len(raw_corpus)} characters\n")

def preprocess_corpus(text):
    """
    Tokenizes text into sentences and words, adds start/end tokens.
    """
    sentences = nltk.sent_tokenize(text.lower()) # Convert to lowercase for consistency
    processed_sentences = []
    for sent in sentences:
        # Tokenize words, filter out non-alphabetic tokens (simple cleanup)
        words = [word for word in nltk.word_tokenize(sent) if word.isalpha()]
        if words: # Ensure the sentence is not empty after filtering
            processed_sentences.append(['<s>'] + words + ['</s>'])
    return processed_sentences

processed_sentences = preprocess_corpus(raw_corpus)

print(f"Number of processed sentences: {len(processed_sentences)}")
print("First 3 processed sentences:")
for i, s in enumerate(processed_sentences[:3]):
    print(f"  {i+1}: {s}")
print("\n")

# Split into training and testing sets (80/20 split)
random.seed(42) # for reproducibility
random.shuffle(processed_sentences)

train_size = int(0.8 * len(processed_sentences))
train_sentences = processed_sentences[:train_size]
test_sentences = processed_sentences[train_size:]

# Flatten lists of words for unigram counts
train_words = [word for sent in train_sentences for word in sent]
test_words = [word for sent in test_sentences for word in sent]

print(f"Train set size (sentences): {len(train_sentences)}")
print(f"Test set size (sentences): {len(test_sentences)}")
print(f"Total words in train set (including <s>, </s>): {len(train_words)}\n")

# --- Q6: Estimate Unigram and Bigram Probabilities (MLE) ---

print("--- Q6: Unigram and Bigram Probabilities (MLE) ---")

def estimate_unigram_probabilities(words):
    """
    Estimates unigram probabilities using Maximum Likelihood Estimation.
    Returns a dictionary of {word: probability}.
    """
    word_counts = Counter(words)
    total_tokens = len(words) # Count all tokens for the denominator
    unigram_probs = {word: count / total_tokens for word, count in word_counts.items()}
    return unigram_probs, word_counts, total_tokens

unigram_probs, unigram_counts, total_train_tokens = estimate_unigram_probabilities(train_words)

print(f"Top 10 Unigram Probabilities (MLE) from training set:")
for word, prob in sorted(unigram_probs.items(), key=lambda item: item[1], reverse=True)[:10]:
    print(f"  P('{word}') = {prob:.6f}")
print("\n")

def estimate_bigram_probabilities(sentences, laplace_smoothing=0):
    """
    Estimates bigram probabilities P(wi | wi-1) using MLE or Laplace smoothing.
    Returns a dictionary of { (w_prev, w_curr): probability }.
    Also returns counts for perplexity calculation.
    """
    bigram_counts = defaultdict(lambda: defaultdict(int))
    unigram_counts_for_bigram_denom = defaultdict(int) # Counts of w_prev

    all_words = []
    for sent in sentences:
        all_words.extend(sent)
        for i in range(len(sent) - 1):
            w_prev = sent[i]
            w_curr = sent[i+1]
            bigram_counts[w_prev][w_curr] += 1
            unigram_counts_for_bigram_denom[w_prev] += 1

    # Get vocabulary from all words in the training set
    vocabulary = set(all_words)
    vocab_size = len(vocabulary)

    bigram_probs = defaultdict(lambda: defaultdict(float))

    for w_prev, next_words_counts in bigram_counts.items():
        denominator = unigram_counts_for_bigram_denom[w_prev] + laplace_smoothing * vocab_size
        for w_curr in next_words_counts:
            numerator = next_words_counts[w_curr] + laplace_smoothing
            bigram_probs[w_prev][w_curr] = numerator / denominator

        # For unseen words following w_prev (if smoothing is applied)
        if laplace_smoothing > 0:
            for w_curr in vocabulary: # Iterate over all possible words in vocabulary
                if w_curr not in next_words_counts: # If this bigram was unseen
                    numerator = laplace_smoothing
                    bigram_probs[w_prev][w_curr] = numerator / denominator

    return bigram_probs, bigram_counts, unigram_counts_for_bigram_denom, vocabulary, vocab_size

# MLE Bigram Model (Laplace smoothing = 0)
mle_bigram_probs, mle_bigram_counts, mle_unigram_denom_counts, vocabulary_mle, vocab_size_mle = \
    estimate_bigram_probabilities(train_sentences, laplace_smoothing=0)

print("Top 5 Bigram Probabilities (MLE) from training set:")
# Extract some bigrams to display
sampled_bigrams = []
for w_prev, next_probs in mle_bigram_probs.items():
    for w_curr, prob in next_probs.items():
        if prob > 0: # Only show existing bigrams for MLE
            sampled_bigrams.append(((w_prev, w_curr), prob))
    if len(sampled_bigrams) > 100: break # Limit for efficiency
sampled_bigrams.sort(key=lambda x: x[1], reverse=True)

for (w_prev, w_curr), prob in sampled_bigrams[:5]:
    print(f"  P('{w_curr}' | '{w_prev}') = {prob:.6f}")
print("\n")

# --- Q7: Add-1 (Laplace) Smoothing for Bigram Model ---

print("--- Q7: Add-1 (Laplace) Smoothing ---")

laplace_bigram_probs, laplace_bigram_counts, laplace_unigram_denom_counts, vocabulary_laplace, vocab_size_laplace = \
    estimate_bigram_probabilities(train_sentences, laplace_smoothing=1)

print("Example of Laplace Smoothed Bigram Probabilities:")
# Find a common word and an uncommon word to demonstrate
example_word_prev_common = 'the'
example_word_prev_uncommon = 'trigrams'

print(f"  P('dog' | '{example_word_prev_common}') (MLE): ", \
      mle_bigram_probs[example_word_prev_common].get('dog', 0.0))
print(f"  P('dog' | '{example_word_prev_common}') (Laplace): ", \
      laplace_bigram_probs[example_word_prev_common].get('dog', 0.0))

# Demonstrate an unseen bigram
unseen_bigram_prev = 'apple' # Assuming 'apple' is not in our corpus as w_prev
unseen_bigram_curr = 'banana'

# To demonstrate, we need an `w_prev` that exists in the vocabulary
# but the bigram `(w_prev, w_curr)` is unseen.
# Let's pick a valid w_prev from the vocabulary to demonstrate unseen P(w_curr | w_prev)
existing_word_prev = 'the' # A common word
unseen_word_curr_after_existing = 'apple' # Assume 'apple' is not a successor of 'the' in training data
if unseen_word_curr_after_existing not in vocabulary_laplace: # Ensure it's in vocabulary if we want to smooth it
    vocabulary_laplace.add(unseen_word_curr_after_existing) # Temporarily add for demonstration
    vocab_size_laplace = len(vocabulary_laplace)

# Recalculate if vocab size changed for demonstration
if vocab_size_laplace != len(vocabulary_laplace):
    laplace_bigram_probs, _, _, _, _ = estimate_bigram_probabilities(train_sentences, laplace_smoothing=1)

mle_prob_unseen = mle_bigram_probs[existing_word_prev].get(unseen_word_curr_after_existing, 0.0)
laplace_prob_unseen = laplace_bigram_probs[existing_word_prev].get(unseen_word_curr_after_existing, 0.0)

print(f"  P('{unseen_word_curr_after_existing}' | '{existing_word_prev}') (MLE): {mle_prob_unseen:.6f}")
print(f"  P('{unseen_word_curr_after_existing}' | '{existing_word_prev}') (Laplace): {laplace_prob_unseen:.6f}")

print("\nBriefly Answer:")
print("Why is raw MLE insufficient for unseen bigrams? Raw MLE assigns a probability of 0 to any bigram that did not appear in the training data. This is problematic because even if a bigram is unseen in training, it might still occur in test data. A 0 probability leads to an undefined (infinite) perplexity, making the model unable to generalize.")
print("What problem does Laplace smoothing solve? Laplace (add-1) smoothing addresses the zero-frequency problem by adding 1 to all observed counts and adding the vocabulary size to the denominator. This ensures that every possible bigram, even unseen ones, gets a non-zero probability. While it might slightly underestimate the probabilities of seen bigrams, it prevents the model from assigning 0 probability to unseen bigrams, leading to more robust probability estimates and finite perplexity.")
print("\n")

# --- Q8: Compute Perplexity ---

print("--- Q8: Perplexity Calculation ---")

def calculate_perplexity(model_type, model_probs, n_gram, test_sentences, vocab_size, unigram_counts=None, total_train_tokens=None):
    """
    Computes the perplexity of a language model on a test set.
    model_type: 'unigram', 'bigram', 'trigram'
    model_probs: dictionary of probabilities (e.g., unigram_probs, laplace_bigram_probs)
    n_gram: 1 for unigram, 2 for bigram, 3 for trigram
    test_sentences: list of sentences from the test set.
    vocab_size: size of the vocabulary (used for handling unknown words or smoothing implicitly for log prob).
    unigram_counts, total_train_tokens: needed for unigram log prob calculation if model_type is 'unigram'
    """
    log_prob_sum = 0.0
    num_words = 0 # Count of actual words (excluding <s> tokens, but including </s> for testing)

    # For unigram, we need to adapt num_words calculation
    if model_type == 'unigram':
        for sentence in test_sentences:
            for word in sentence:
                if word != '<s>':
                    num_words += 1
                    prob = model_probs.get(word, 1e-10) # Assign a very small prob for unseen unigrams
                    log_prob_sum += math.log(prob, 2)
    elif model_type == 'bigram':
        for sentence in test_sentences:
            for i in range(len(sentence) - 1):
                w_prev = sentence[i]
                w_curr = sentence[i+1]
                if w_curr != '<s>': # Count actual words, not the starting <s> token for n-gram context
                    num_words += 1

                # Get bigram probability P(w_curr | w_prev)
                prob = model_probs[w_prev].get(w_curr, 0.0) # Should be non-zero with Laplace
                if prob == 0.0:
                    prob = 1e-10 # Fallback for extremely rare edge cases or out-of-vocabulary words
                log_prob_sum += math.log(prob, 2)

    elif model_type == 'trigram':
        for sentence in test_sentences:
            for i in range(len(sentence) - 2):
                w_prev2 = sentence[i]
                w_prev1 = sentence[i+1]
                w_curr = sentence[i+2]
                if w_curr != '<s>': # Count actual words, not starting <s>
                    num_words += 1

                # Get trigram probability P(w_curr | w_prev1, w_prev2)
                # Trigram model_probs would be dict[w_prev2][w_prev1][w_curr]
                prob = model_probs[w_prev2][w_prev1].get(w_curr, 0.0)
                if prob == 0.0:
                    prob = 1e-10 # Fallback
                log_prob_sum += math.log(prob, 2)
    else:
        raise ValueError("Invalid model_type")

    if num_words == 0: # Avoid division by zero
        return float('inf')

    # Perplexity = 2 ^ (- (1/N) * sum(log P(w_i))) where log is base 2
    perplexity = 2 ** (-log_prob_sum / num_words)
    return perplexity

# Perplexity for Unigram Model
unigram_perplexity = calculate_perplexity(
    'unigram', unigram_probs, 1, test_sentences, len(unigram_counts),
    unigram_counts=unigram_counts, total_train_tokens=total_train_tokens
)

# Perplexity for Bigram Model (with Laplace smoothing)
bigram_laplace_perplexity = calculate_perplexity(
    'bigram', laplace_bigram_probs, 2, test_sentences, vocab_size_laplace
)

print(f"Unigram Model Perplexity on Test Set: {unigram_perplexity:.2f}")
print(f"Bigram Model (Laplace Smoothed) Perplexity on Test Set: {bigram_laplace_perplexity:.2f}\n")

print("Perplexity Report (Q8):")
print("| Model                 | Perplexity |")
print("|-----------------------|------------|")
print(f"| Unigram               | {unigram_perplexity:<10.2f} |")
print(f"| Bigram (Laplace)      | {bigram_laplace_perplexity:<10.2f} |")
print("\n")

print("Explanation for Q8:")
print(f"Why bigram model has {'higher' if bigram_laplace_perplexity > unigram_perplexity else 'lower' if bigram_laplace_perplexity < unigram_perplexity else 'similar'} perplexity than the unigram model in this instance: Perplexity is a measure of how well a probability model predicts a sample. Lower perplexity indicates a better model. Theoretically, a bigram model, by incorporating context (P(wi | wi-1)), should make more informed predictions and achieve lower perplexity than a unigram model which only considers individual word frequencies. However, in this specific execution, the Laplace-smoothed bigram model exhibits {'higher' if bigram_laplace_perplexity > unigram_perplexity else 'lower' if bigram_laplace_perplexity < unigram_perplexity else 'similar'} perplexity than the unigram model. This unexpected result can occur due to factors such as:")
print("  - **Sparsity**: Even with a large corpus, specific bigrams might still be very rare or unseen in the training data. When these unseen bigrams appear in the test set, assigning them a small smoothed probability (e.g., through add-1 smoothing) can still lead to a high perplexity if there are many such instances.")
print("  - **Over-smoothing**: Add-1 (Laplace) smoothing is a very simplistic technique. While it addresses zero probabilities, it can significantly 'flatten' the probability distribution by assigning a small, uniform probability mass to all unseen events. For frequently occurring bigrams, add-1 smoothing can dilute their probabilities too much, reducing the model's confidence in observed patterns. This over-smoothing effect can sometimes outweigh the benefits of incorporating context, leading to a worse performance (higher perplexity) compared to a simpler unigram model, especially if the bigram dependencies are not strong enough to overcome the smoothing penalty.")
print("\n")

# --- Q9: Generate Random Sentences from Bigram Model ---

print("--- Q9: Generate Random Sentences (Bigram Model) ---")

def generate_sentence(bigram_model_probs, max_length=20):
    """
    Generates a random sentence using ancestral sampling from a bigram model.
    """
    sentence = ['<s>']
    current_word = '<s>'

    while current_word != '</s>' and len(sentence) < max_length:
        # Get possible next words and their probabilities given the current_word
        next_word_probs = bigram_model_probs[current_word]

        if not next_word_probs: # If no successors for current_word (should not happen with smoothing)
            break

        # Create lists for words and their probabilities for random.choices
        words = list(next_word_probs.keys())
        probabilities = list(next_word_probs.values())

        # Handle case where probabilities don't sum to 1 due to floating point or other issues
        # Normalize if necessary (e.g., if some very small non-smoothed probabilities led to 0)
        prob_sum = sum(probabilities)
        if prob_sum == 0:
             # Fallback: if no next words have non-zero probability (problematic case)
             # this should not happen with Laplace smoothing if </s> is in vocab
             break
        probabilities = [p / prob_sum for p in probabilities] # Normalize

        # Sample the next word based on probabilities
        next_word_choices = random.choices(words, weights=probabilities, k=1)
        current_word = next_word_choices[0]
        sentence.append(current_word)

    return ' '.join(sentence).replace('<s> ', '').replace(' </s>', '')

print("Generated 5 random sentences from the Laplace-smoothed Bigram Model:")
for i in range(5):
    sentence = generate_sentence(laplace_bigram_probs)
    print(f"  {i+1}: {sentence}")
print("\n")

# --- Q10: Trigram Model with Laplace Smoothing and Perplexity ---

print("--- Q10: Trigram Model with Laplace Smoothing ---")

def estimate_trigram_probabilities(sentences, laplace_smoothing=0):
    """
    Estimates trigram probabilities P(wi | wi-1, wi-2) using MLE or Laplace smoothing.
    Returns a dictionary of { (w_prev2, w_prev1): {w_curr: probability} }.
    """
    trigram_counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    bigram_counts_for_trigram_denom = defaultdict(lambda: defaultdict(int)) # Counts of (w_prev2, w_prev1)

    all_words = []
    for sent in sentences:
        all_words.extend(sent)
        for i in range(len(sent) - 2):
            w_prev2 = sent[i]
            w_prev1 = sent[i+1]
            w_curr = sent[i+2]
            trigram_counts[w_prev2][w_prev1][w_curr] += 1
            bigram_counts_for_trigram_denom[w_prev2][w_prev1] += 1

    vocabulary = set(all_words)
    vocab_size = len(vocabulary)

    trigram_probs = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))

    for w_prev2, bigrams_dict in trigram_counts.items():
        for w_prev1, next_words_counts in bigrams_dict.items():
            denominator = bigram_counts_for_trigram_denom[w_prev2][w_prev1] + laplace_smoothing * vocab_size
            if denominator == 0: # Avoid division by zero for cases where (w_prev2, w_prev1) never occurred
                continue

            for w_curr in next_words_counts:
                numerator = next_words_counts[w_curr] + laplace_smoothing
                trigram_probs[w_prev2][w_prev1][w_curr] = numerator / denominator

            # For unseen words following (w_prev2, w_prev1) if smoothing is applied
            if laplace_smoothing > 0:
                for w_curr in vocabulary:
                    if w_curr not in next_words_counts:
                        numerator = laplace_smoothing
                        trigram_probs[w_prev2][w_prev1][w_curr] = numerator / denominator

    return trigram_probs, vocabulary, vocab_size

# Build Trigram Model with Laplace Smoothing
laplace_trigram_probs, vocabulary_trigram, vocab_size_trigram = \
    estimate_trigram_probabilities(train_sentences, laplace_smoothing=1)

# Perplexity for Trigram Model (with Laplace smoothing)
trigram_laplace_perplexity = calculate_perplexity(
    'trigram', laplace_trigram_probs, 3, test_sentences, vocab_size_trigram
)

print(f"Trigram Model (Laplace Smoothed) Perplexity on Test Set: {trigram_laplace_perplexity:.2f}\n")

print("Combined Perplexity Report (Q10):")
print("| Model                 | Perplexity |")
print("|-----------------------|------------|")
print(f"| Unigram               | {unigram_perplexity:<10.2f} |")
print(f"| Bigram (Laplace)      | {bigram_laplace_perplexity:<10.2f} |")
print(f"| Trigram (Laplace)     | {trigram_laplace_perplexity:<10.2f} |")
print("\n")

print("Discussion on Trigram Perplexity vs. Bigram Perplexity (Q10):")
print(f"Does your trigram model have higher perplexity than the bigram model despite being trained on the same corpus? In this specific example, the trigram model's perplexity ({trigram_laplace_perplexity:.2f}) is {('higher' if trigram_laplace_perplexity > bigram_laplace_perplexity else 'lower' if trigram_laplace_perplexity < bigram_laplace_perplexity else 'similar')} than the bigram model's ({bigram_laplace_perplexity:.2f}). This can happen due to several factors:")
print("  - **Sparsity**: As n-gram length increases (from bigram to trigram), the number of unique n-grams grows exponentially. This leads to a severe sparsity problem: many trigrams that might occur in the test set are likely to be unseen in the training set, especially with a small corpus. An unseen event receives a very low probability (even with smoothing), which can dramatically increase perplexity.")
print("  - **Smoothing**: Laplace smoothing (add-1) is a very simplistic technique. While it addresses zero probabilities, it tends to over-smooth, especially for higher-order n-grams like trigrams. By adding 1 to every count, it significantly reduces the probabilities of frequently observed n-grams and inflates the probabilities of unseen ones. For a small corpus, this over-smoothing can degrade the model's performance by making it less confident about actually observed patterns.")
print("  - **Effect of Conditioning on a Longer History**: Theoretically, a trigram model, by conditioning on two preceding words (P(wi | wi-1, wi-2)), should capture more context and thus be more accurate than a bigram model (P(wi | wi-1)). This increased context *should* lead to lower perplexity. However, this benefit is often outweighed by the sparsity and over-smoothing issues when the training corpus is small. With a sufficiently large corpus, the benefits of longer context usually manifest as lower perplexity. For small corpora, the data is too sparse to reliably estimate the probabilities of complex trigrams, making the bigram model sometimes perform better or similarly, especially with simple smoothing techniques.")
print("In summary, while trigram models theoretically offer better context, their performance can suffer with small datasets due to data sparsity and the limitations of simple smoothing methods like add-1, potentially leading to higher (or not significantly lower) perplexity compared to bigram models.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


--- Corpus Preprocessing ---
Raw Corpus Length: 887071 characters

Number of processed sentences: 7455
First 3 processed sentences:
  1: ['<s>', 'emma', 'by', 'jane', 'austen', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with', 'a', 'comfortable', 'home', 'and', 'happy', 'disposition', 'seemed', 'to', 'unite', 'some', 'of', 'the', 'best', 'blessings', 'of', 'existence', 'and', 'had', 'lived', 'nearly', 'years', 'in', 'the', 'world', 'with', 'very', 'little', 'to', 'distress', 'or', 'vex', 'her', '</s>']
  2: ['<s>', 'she', 'was', 'the', 'youngest', 'of', 'the', 'two', 'daughters', 'of', 'a', 'most', 'affectionate', 'indulgent', 'father', 'and', 'had', 'in', 'consequence', 'of', 'her', 'sister', 'marriage', 'been', 'mistress', 'of', 'his', 'house', 'from', 'a', 'very', 'early', 'period', '</s>']
  3: ['<s>', 'her', 'mother', 'had', 'died', 'too', 'long', 'ago', 'for', 'her', 'to', 'have', 'more', 'than', 'an', 'indistinct', 'remembrance', 'o

## Language Model Perplexity Report (Q10)

### Combined Perplexity Values

| Model                 | Perplexity |
|-----------------------|------------|
| Unigram               | 613.78     |
| Bigram (Laplace)      | 1904.80    |
| Trigram (Laplace)     | 2715.15    |


### Discussion on Trigram vs. Bigram Perplexity

In this analysis, the Laplace-smoothed trigram model yielded a perplexity of **2715.15**, which is *higher* than the bigram model's perplexity of **1904.80**. This result, while counter-intuitive from a theoretical standpoint (as longer contexts should generally improve prediction), can be attributed to the following key factors:

1.  **Sparsity**: As the order of the N-gram model increases from bigram to trigram, the number of unique sequences of words grows exponentially. Even with a moderately sized corpus like 'austen-emma.txt', many possible trigrams will simply not appear in the training data. When these unseen trigrams occur in the test set, they are assigned a very small, smoothed probability. This severe sparsity means the model has very little reliable data to estimate trigram probabilities accurately, leading to poorer performance on unseen sequences and thus higher perplexity.

2.  **Smoothing (Over-smoothing)**: Add-1 (Laplace) smoothing, while effective at preventing zero probabilities, is a very basic technique. For higher-order N-grams like trigrams, where the vocabulary size is large and the number of unseen combinations is vast, adding '1' to every count (and the vocabulary size to the denominator) leads to significant *over-smoothing*. This dilutes the probabilities of frequently observed trigrams, making the model less confident about actual linguistic patterns, and assigns disproportionately high probability mass to unobserved events. This can degrade the model's predictive accuracy, outweighing any benefits of increased context.

3.  **Effect of Conditioning on a Longer History**: Theoretically, a trigram model, by considering two preceding words (`P(w_i | w_i-1, w_i-2)`), captures more contextual information than a bigram model (`P(w_i | w_i-1)`). This enhanced context *should* lead to more accurate predictions and lower perplexity. However, this advantage is heavily dependent on the availability of sufficient training data. In scenarios with insufficient data, the benefits of a longer history are nullified by the overwhelming problems of sparsity and over-smoothing. For the given corpus size and the simplicity of add-1 smoothing, the data is too sparse to effectively estimate the complex conditional probabilities of trigrams, preventing the theoretical gains from materializing and instead leading to higher perplexity.

**Conclusion**: While trigram models inherently aim for a deeper understanding of linguistic context, their practical performance can suffer significantly with limited datasets when coupled with simple smoothing techniques. The challenges of data sparsity and over-smoothing for higher-order N-grams can lead to higher perplexity values, indicating a less effective predictive model compared to simpler bigram models in such conditions. More advanced smoothing methods are typically required to harness the full potential of trigram (and higher-order) language models.